In [38]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [22]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [23]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [24]:
# features = [
#     'EventTimeStamp',
#     'EquipmentID',
#     'spn',
#     'fmi',
#     'active',
#     'Severity_Level',
#     'BarometricPressure',
#     'EngineCoolantTemperature',
#     'EngineLoad',
#     'EngineOilPressure',
#     'EngineOilTemperature',
#     'EngineRpm',
#     'FuelRate',
#     'FuelTemperature',
#     'IntakeManifoldTemperature',
#     'Speed',
#     'Throttle',
#     'TurboBoostPressure'
# ]
# len(features)

In [25]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

## Identify features for imputing missing values

In [27]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [29]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# low_nan_numeric_pipe = Pipeline(
#     steps=[
#         ('scaler', StandardScaler()),
#         ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
#     ]
# )

numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [30]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        # ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns),
        ('medium_nan_numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [31]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(32,32,32)
        ))
    ]
)

In [32]:
pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('medium_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('medium_nan_numeric_imputer',
                                                                   IterativeImputer(max_iter=20,
                                                                                    random_state=30))]),
                                                  ['BarometricPressure',
                                                   'EngineCoolantTemperature',
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model', MLPClassifier(hidden_layer_sizes=(32, 32, 32)))])

## Adjust Threshold

In [34]:
y_pred_prob_val = pipe.predict_proba(X_val)[:,1]

In [51]:
labels=[0,1,2]

candidate_thresholds = np.arange(
    start=0.01,
    stop=0.925,
    step=0.01
)

thresholds_df = pd.DataFrame({'threshold': candidate_thresholds})
thresholds_df['f1'] = thresholds_df['threshold'].apply(lambda x: f1_score(y_true=y_val, y_pred=y_pred_prob_val >= x, labels=labels, average='macro'))
thresholds_df.sort_values(by='f1', ascending=False, inplace=True)
thresholds_df.head()

,threshold,f1
23,0.24,0.427089
19,0.20,0.426567
22,0.23,0.426488
21,0.22,0.426488
26,0.27,0.425779


## Compare training and testing

In [52]:
threshold = thresholds_df['threshold'].iloc[0]

y_pred_proba_train = pipe.predict_proba(X_train)[:,1]
y_pred_proba_test = pipe.predict_proba(X_test)[:,1]

y_pred_train = y_pred_proba_train >= threshold
y_pred_test = y_pred_proba_test >= threshold

In [53]:
training_cr = classification_report(
    y_true=y_train,
    y_pred=y_pred_train,
    digits=6
)
print(str(training_cr))

training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train,
    labels=labels
)
print(training_cm)

training_mcm = multilabel_confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train,
    labels=labels
)
print(training_mcm)

              precision    recall  f1-score   support

           0   0.998787  0.999851  0.999319    950562
           1   0.547965  0.471840  0.507061       799
           2   0.000000  0.000000  0.000000       901

    accuracy                       0.998462    952262
   macro avg   0.515584  0.490563  0.502127    952262
weighted avg   0.997464  0.998462  0.997960    952262

[[950420    142      0]
 [   422    377      0]
 [   732    169      0]]
[[[   546   1154]
  [   142 950420]]

 [[951152    311]
  [   422    377]]

 [[951361      0]
  [   901      0]]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [54]:
testing_cr = classification_report(
    y_true=y_test,
    y_pred=y_pred_test,
    digits=6
)
print(str(testing_cr))

testing_cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
print(testing_cm)

testing_mcm = multilabel_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test,
    labels=labels
)
print(testing_mcm)

              precision    recall  f1-score   support

           0   0.998714  0.999825  0.999269    211236
           1   0.447552  0.359551  0.398754       178
           2   0.000000  0.000000  0.000000       200

    accuracy                       0.998341    211614
   macro avg   0.482089  0.453125  0.466008    211614
weighted avg   0.997306  0.998341  0.997819    211614

[[211199     37      0]
 [   114     64      0]
 [   158     42      0]]
[[[   106    272]
  [    37 211199]]

 [[211357     79]
  [   114     64]]

 [[211414      0]
  [   200      0]]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
